# Agentic AI for Prescriptive Maintenance

**Course:** AI in Aviation — Day 2 Demo  
**Stack:** Ollama (qwen2.5:3b) + LangChain (tool calling) + LangGraph (ReAct orchestration)  \n
**Objective:** Build an autonomous agent that combines a predictive RUL model with domain knowledge (maintenance manual) to produce actionable prescriptive maintenance recommendations.

**Prerequisite:** Install [`uv`](https://docs.astral.sh/uv/) and [`Ollama`](https://ollama.com/) once per machine.

```bash
# macOS / Linux — install uv
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows (PowerShell) — install uv
# powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

# Verify
uv --version
```

**Install Ollama:**

| OS | Instructions |
|---|---|
| **macOS** | Download from [ollama.com/download/mac](https://ollama.com/download/mac) or run `brew install ollama` |
| **Windows** | Download installer from [ollama.com/download/windows](https://ollama.com/download/windows) |
| **Linux** | Run `curl -fsSL https://ollama.com/install.sh \| sh` (see [ollama.com/download/linux](https://ollama.com/download/linux)) |

**After installing Ollama, verify it's running and pull the model:**

```bash
# Verify Ollama is running
ollama --version

# Pull the model (run once, ~1.9 GB download)
ollama pull qwen2.5:3b

# Verify model is available
ollama list | grep qwen2.5
```

**About Qwen 2.5:** This model is developed by Alibaba's Tongyi Lab in China. It's part of the broader Chinese AI ecosystem that has produced some of the most competitive open-weight models globally — rivaling Western models from Google (Gemma), Meta (Llama), and Mistral. The 3B variant is small enough to run on modest hardware (even an 8 GB laptop) while still supporting reliable tool calling and reasoning. For more: [qwen.ai](https://qwen.ai/research).

**Set up the project environment** (run once per machine, from the project root):

```bash
# Initialize the uv project
uv init --python 3.11

# Add runtime dependencies
uv add pandas numpy scikit-learn joblib

# Add agentic AI dependencies
uv add langchain langchain-core langchain-openai langgraph

# Add notebook dependencies
uv add jupyterlab ipykernel ipython

# Register the kernel
uv run python -m ipykernel install --user --name ai_in_aviation --display-name "Python (ai_in_aviation)"
```

**Launch Jupyter:**

```bash
uv run jupyter lab
```

**Before running this notebook:**

1. Make sure Ollama is running (the Ollama menu bar icon should show "Ollama is running" on macOS).
2. Make sure the model is pulled: `ollama pull qwen2.5:3b`.\n
3. Open this notebook in Jupyter and select the `Python (ai_in_aviation)` kernel.

**Reproduce on another machine:**

```bash
uv sync
ollama pull qwen2.5:3b
uv run jupyter lab
```

---

## From Prediction to Prescription

On Day 1, we built a model that **predicts** how many cycles remain before engine failure. That is valuable, but it stops short of telling us **what to do** about it.

Today, we will go one step further. We will build an **agentic AI system** that:

1. **Queries** the Day 1 RUL model for engine health status.
2. **Reads** an engine maintenance manual for procedures and thresholds.
3. **Reasons** about the situation using a ReAct (Reason + Act) loop.
4. **Recommends** a prescriptive maintenance action.

This is the pattern: **Harness (orchestrator) + LLM (reasoning) + Tools (deterministic actions)**.

The LLM provides the reasoning and natural language understanding. The tools provide reliable, deterministic computations (RUL prediction, manual lookup). The orchestrator (LangGraph) manages the loop between them.

All of this runs locally on your machine — no cloud API, no internet connection required beyond the initial model download.

## 1. Setup and Imports

In [1]:
# Standard libraries
import pandas as pd
import numpy as np
import joblib
import os
import re
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.preprocessing import StandardScaler

# LangChain & LangGraph
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

# Display utilities
from IPython.display import display, Markdown as display_md

# Pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
pd.set_option('display.precision', 2)

## 2. Pre-Flight: Check Ollama Connectivity

Before we proceed, let's verify that Ollama is running and the model is available. The cell below will ping the local Ollama server.

In [2]:
import urllib.request
import json

# Local Ollama endpoint
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_MODEL = "qwen2.5:3b"

# Check if Ollama is running
try:
    req = urllib.request.Request(
        f"{OLLAMA_BASE_URL}/models",
        headers={"User-Agent": "ai-in-aviation-demo"},
        method="GET"
    )
    with urllib.request.urlopen(req, timeout=10) as response:
        models = json.loads(response.read().decode())
        model_names = [m['id'] for m in models.get('data', [])]
        
        # Check if our model is available
        model_found = any(OLLAMA_MODEL.split(':')[0] in m for m in model_names)
        
        if model_found:
            print(f"✅ Ollama is running. Model '{OLLAMA_MODEL}' is available.")
        else:
            print(f"✅ Ollama is running, but '{OLLAMA_MODEL}' not found.")
            print(f"   Available models: {', '.join(model_names[:5])}")
            print(f"   Run: ollama pull {OLLAMA_MODEL}")
            raise ConnectionError(f"Model '{OLLAMA_MODEL}' not found. Run: ollama pull {OLLAMA_MODEL}")
            
except urllib.error.URLError:
    print("❌ Ollama is not running.")
    print("   Start Ollama (look for the Ollama icon in your menu bar).")
    print("   Or run: ollama serve")
    raise ConnectionError("Ollama is not running. Start Ollama before proceeding.")

print(f"\nActive configuration:")
print(f"  Base URL: {OLLAMA_BASE_URL}")
print(f"  Model:    {OLLAMA_MODEL}")

✅ Ollama is running. Model 'qwen2.5:3b' is available.

Active configuration:
  Base URL: http://localhost:11434/v1
  Model:    qwen2.5:3b


## 3. Load the Day 1 RUL Model

We will load the trained model artifact from Day 1. This model predicts Remaining Useful Life (RUL) from engine sensor data using a sliding window feature extraction pipeline.

The saved file contains:
- The trained regression model
- The feature scaler (StandardScaler)
- The sliding window size used during training
- Model metadata (type, RMSE, PHM score)

In [3]:
# Load the saved model artifact
model_path = '../assets/models/rul_model.joblib'
loaded = joblib.load(model_path)

rul_model = loaded['model']
rul_scaler = loaded['scaler']
window_size = loaded['window_size']
model_type = loaded['model_type']
model_rmse = loaded['rmse']
model_phm = loaded['phm_score']

print(f"✅ Model loaded successfully from {model_path}")
print(f"  Model type:    {model_type}")
print(f"  Window size:   {window_size} cycles")
print(f"  Test RMSE:     {model_rmse:.2f} cycles")
print(f"  Test PHM Score: {model_phm:.2f}")

✅ Model loaded successfully from ../assets/models/rul_model.joblib
  Model type:    Gradient Boosting
  Window size:   50 cycles
  Test RMSE:     19.10 cycles
  Test PHM Score: 630.70


### Feature Extraction Pipeline

To make a prediction, we need to transform raw sensor data into the same 84 features the model was trained on. This is the **sliding window** approach from Day 1:

For each of 21 sensors, compute 4 rolling statistics (mean, std, min, max) over the last N cycles → 21 × 4 = **84 features**.

In [4]:
def extract_features(df, window_size):
    """Extract sliding window features from raw sensor data.
    
    For each cycle of each engine, compute rolling statistics over the 
    previous window_size cycles. Returns one row per cycle.
    
    Args:
        df: DataFrame with raw sensor data (columns: unit_number, cycle, sensor_1..sensor_21).
        window_size: Number of cycles to look back for rolling statistics.
    
    Returns:
        DataFrame with extracted features, one row per cycle.
    """
    sensor_cols = [c for c in df.columns if c.startswith('sensor_')]
    feature_frames = []
    
    for unit_id, group in df.groupby('unit_number'):
        group = group.sort_values('cycle').copy()
        rolling = group[sensor_cols].rolling(window=window_size, min_periods=1)
        roll_mean = rolling.mean()
        roll_std = rolling.std().fillna(0)
        roll_min = rolling.min()
        roll_max = rolling.max()
        
        features = pd.DataFrame()
        for col in sensor_cols:
            features[f'{col}_mean'] = roll_mean[col].values
            features[f'{col}_std'] = roll_std[col].values
            features[f'{col}_min'] = roll_min[col].values
            features[f'{col}_max'] = roll_max[col].values
        
        features['unit_number'] = group['unit_number'].values
        features['cycle'] = group['cycle'].values
        feature_frames.append(features)
    
    return pd.concat(feature_frames, ignore_index=True)


def predict_rul_for_engine(engine_data_df, model, scaler, window_size):
    """Predict RUL for a single engine from its raw sensor data.
    
    Args:
        engine_data_df: DataFrame with raw sensor data for ONE engine.
        model: Trained sklearn regressor.
        scaler: Fitted StandardScaler.
        window_size: Sliding window size.
    
    Returns:
        Predicted RUL (float, in cycles) for the last available cycle.
    """
    features = extract_features(engine_data_df, window_size)
    feature_cols = [c for c in features.columns if c not in ['unit_number', 'cycle']]
    
    # Take the last cycle
    last_row = features.tail(1)[feature_cols].values
    last_row_scaled = scaler.transform(last_row)
    
    pred = model.predict(last_row_scaled)[0]
    # RUL cannot be negative; cap at realistic max
    return max(0.0, min(pred, 125.0))

### Verify the Model Works

Let's run a quick test prediction using data from the C-MAPSS test set to confirm the pipeline is working.

In [5]:
# Column names for C-MAPSS data
column_names = [
    'unit_number', 'cycle',
    'op_setting_1', 'op_setting_2', 'op_setting_3',
    'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5',
    'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10',
    'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15',
    'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20',
    'sensor_21'
]

# Load test data for Engine 1
test_path = '../assets/cmapss/test_FD001.txt'
test_raw = pd.read_csv(test_path, sep=r'\s+', engine='python', header=None, names=column_names)

# Load true RUL values
rul_path = '../assets/cmapss/RUL_FD001.txt'
rul_raw = pd.read_csv(rul_path, sep=r'\s+', engine='python', header=None, names=['rul', 'threshold'])

# Test with Engine 1
engine_1_data = test_raw[test_raw['unit_number'] == 1].copy()
predicted_rul = predict_rul_for_engine(engine_1_data, rul_model, rul_scaler, window_size)
true_rul = rul_raw.iloc[0]['rul']  # Engine 1's true RUL

print(f"Engine 1 verification:")
print(f"  Predicted RUL: {predicted_rul:.1f} cycles")
print(f"  True RUL:      {true_rul} cycles")
print(f"  Absolute error: {abs(predicted_rul - true_rul):.1f} cycles")
print(f"\n✅ Model pipeline verified — predictions are within expected range.")

Engine 1 verification:
  Predicted RUL: 122.2 cycles
  True RUL:      112.0 cycles
  Absolute error: 10.2 cycles

✅ Model pipeline verified — predictions are within expected range.


## 4. Load the Maintenance Manual

Our agent needs access to domain knowledge. We will load a sample Engine Maintenance Program (EMP) document that contains inspection intervals, replacement thresholds, and RUL-based decision matrices.

The agent will use a simple text search tool to look up relevant sections of the manual.

[Sample Engine Maintenance Program](../assets/manuals/engine_maintenance_program.md)

In [6]:
# Load the maintenance manual
manual_path = '../assets/manuals/engine_maintenance_program.md'
with open(manual_path, 'r') as f:
    manual_text = f.read()

# Split manual into sections for targeted lookup
manual_sections = {}
current_section = "Introduction"
current_content = []

for line in manual_text.split('\n'):
    # Detect section headers (## or ###)
    if line.startswith('## ') and not line.startswith('###'):
        if current_content:
            manual_sections[current_section] = '\n'.join(current_content).strip()
        current_section = line.strip('# ').strip()
        current_content = []
    else:
        current_content.append(line)
# Save last section
if current_content:
    manual_sections[current_section] = '\n'.join(current_content).strip()

print(f"✅ Manual loaded: {len(manual_text)} characters, {len(manual_sections)} sections")
print(f"Sections: {', '.join(manual_sections.keys())}")

✅ Manual loaded: 8865 characters, 9 sections
Sections: Introduction, 1. Purpose and Scope, 2. Engine Overview, 3. Inspection Intervals, 4. Component Replacement Thresholds, 5. RUL-Based Maintenance Decision Matrix, 6. Corrective Actions by Fault Mode, 7. Maintenance Documentation Requirements, 8. References


## 5. Define Agent Tools

Now we define the **tools** that our agent can use. Each tool is a Python function decorated with `@tool`. The LLM reads the function's docstring to understand what the tool does and when to call it.

We will define three tools:
1. **`query_rul`** — Predict RUL for a given engine using the Day 1 model.
2. **`lookup_manual`** — Search the maintenance manual for relevant procedures.
3. **`assess_health_status`** — Map an RUL value to a health status category.

In [7]:
@tool
def query_rul(engine_id: int) -> str:
    """Query the Remaining Useful Life (RUL) prediction for a specific engine.
    
    This tool uses the trained machine learning model from Day 1 to predict
    how many operating cycles remain before the engine reaches failure.
    
    Args:
        engine_id: The engine unit number (integer, e.g., 1, 5, 10).
    
    Returns:
        A string with the predicted RUL in cycles and model metadata.
    """
    try:
        engine_data = test_raw[test_raw['unit_number'] == engine_id].copy()
        if engine_data.empty:
            return f"Engine {engine_id} not found in the dataset."
        
        pred = predict_rul_for_engine(engine_data, rul_model, rul_scaler, window_size)
        last_cycle = engine_data['cycle'].max()
        
        return (
            f"Engine {engine_id} RUL Prediction:\n"
            f"  Predicted RUL: {pred:.1f} cycles\n"
            f"  Last observed cycle: {last_cycle}\n"
            f"  Model: {model_type} (RMSE: {model_rmse:.2f})\n"
            f"  Window size: {window_size} cycles"
        )
    except Exception as e:
        return f"Error querying RUL for Engine {engine_id}: {str(e)}"


@tool
def lookup_manual(query: str) -> str:
    """Search the engine maintenance manual for relevant procedures and thresholds.
    
    This tool searches the Engine Maintenance Program (EMP) document for
    sections matching the query keywords. It returns the matching section content.
    
    Useful queries: 'inspection', 'replacement', 'RUL', 'threshold', 
    'HPC', 'fan', 'degradation', 'corrective', 'decision', 'maintenance'.
    
    Args:
        query: A keyword or phrase to search for in the manual.
    
    Returns:
        The content of the most relevant manual section.
    """
    query_lower = query.lower()
    # Tokenize into individual keywords so multi-word queries (e.g. "HPC
    # degradation corrective") still match sections that contain those
    # words, even if not as one continuous phrase.
    query_words = [w for w in re.findall(r'\w+', query_lower) if len(w) > 2]
    
    # Score each section by keyword match
    best_section = None
    best_score = -1
    
    for section_name, content in manual_sections.items():
        section_lower = section_name.lower()
        content_lower = content.lower()
        score = 0
        
        # Whole-phrase match is the strongest signal
        if query_lower in section_lower:
            score += 20
        score += content_lower.count(query_lower) * 5
        
        # Individual keyword matches (handles multi-word natural-language queries)
        for word in query_words:
            if word in section_lower:
                score += 5
            score += content_lower.count(word)
        
        if score > best_score:
            best_score = score
            best_section = section_name
    
    if best_section and best_score > 0:
        content = manual_sections[best_section]
        # Truncate if too long (keep it readable for the LLM)
        if len(content) > 2000:
            content = content[:2000] + "\n... [content truncated]"
        return f"Manual Section: {best_section}\n\n{content}"
    else:
        # Return the RUL decision matrix as default (most useful section)
        default = manual_sections.get('RUL-Based Maintenance Decision Matrix', 
                                       manual_sections.get('5. RUL-Based Maintenance Decision Matrix',
                                                            'No matching section found.'))
        return f"No exact match for '{query}'. Returning RUL decision matrix:\n\n{default}"


@tool
def assess_health_status(rul_cycles: float) -> str:
    """Assess the health status of an engine based on its RUL value.
    
    This tool maps a Remaining Useful Life (RUL) value to a standardized
    health status category and recommended action timeframe.
    
    Args:
        rul_cycles: The predicted RUL in operating cycles (float).
    
    Returns:
        Health status category and recommended action.
    """
    rul = float(rul_cycles)
    
    if rul > 200:
        status = "HEALTHY"
        action = "Continue normal operations. Next scheduled A-Check applies."
    elif rul > 100:
        status = "MONITOR"
        action = "Increase sensor monitoring frequency. Review trend data at next A-Check."
    elif rul > 50:
        status = "WARNING"
        action = "Schedule a B-Check within the next 30 cycles. Prepare component replacement parts."
    elif rul > 30:
        status = "CAUTION"
        action = "Schedule a B-Check immediately. Conduct borescope inspection of HPC and turbine sections."
    elif rul > 15:
        status = "ADVISORY"
        action = "Plan for C-Check or targeted hot-section inspection. Consider engine removal if RUL drops below 20."
    else:
        status = "CRITICAL"
        action = "REMOVE ENGINE FROM SERVICE IMMEDIATELY. Schedule full C-Check or engine swap. Do not dispatch."
    
    return (
        f"Health Assessment for RUL = {rul:.1f} cycles:\n"
        f"  Status:  {status}\n"
        f"  Action:  {action}"
    )


### Test Each Tool Independently

Before wiring the tools to the agent, let's verify each one works correctly on its own.

In [8]:
print("=" * 60)
print("TOOL 1: query_rul(engine_id=1)")
print("=" * 60)
print(query_rul.invoke({"engine_id": 1}))

print("\n" + "=" * 60)
print("TOOL 2: lookup_manual(query='RUL decision')")
print("=" * 60)
print(lookup_manual.invoke({"query": "RUL decision"}))

print("\n" + "=" * 60)
print("TOOL 3: assess_health_status(rul_cycles=35.0)")
print("=" * 60)
print(assess_health_status.invoke({"rul_cycles": 35.0}))

TOOL 1: query_rul(engine_id=1)
Engine 1 RUL Prediction:
  Predicted RUL: 122.2 cycles
  Last observed cycle: 31
  Model: Gradient Boosting (RMSE: 19.10)
  Window size: 50 cycles

TOOL 2: lookup_manual(query='RUL decision')
Manual Section: 5. RUL-Based Maintenance Decision Matrix

The onboard prognostics system provides a Remaining Useful Life (RUL) estimate in operating cycles. The following decision matrix maps RUL ranges to required maintenance actions.

| RUL Range | Health Status | Required Action | Timeframe |
|---|---|---|---|
| **RUL > 200 cycles** | **Healthy** | Continue normal operations. Next scheduled A-Check applies. | — |
| **RUL 100–200 cycles** | **Monitor** | Increase sensor monitoring frequency to every flight. Review trend data at next A-Check. | Next A-Check |
| **RUL 50–100 cycles** | **Warning** | Schedule a B-Check within the next 30 cycles. Prepare component replacement parts based on trending. | Within 30 cycles |
| **RUL 30–50 cycles** | **Caution** | Schedule

## 6. Build the Agentic AI Orchestrator

Now we connect the tools to the LLM using LangGraph's `create_react_agent`. This creates a **ReAct loop** (Reason + Act):

1. **Thought:** The LLM reasons about what to do next.
2. **Action:** The LLM calls a tool with specific arguments.
3. **Observation:** The tool returns a result.
4. **Repeat** until the LLM has enough information to produce a final answer.

This is the core of agentic AI — the LLM doesn't just generate text; it *acts* by calling tools, observes results, and iterates.

In [9]:
# Create the LLM connection to local Ollama
llm = ChatOpenAI(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    api_key="not-needed",  # No API key required for local Ollama
    temperature=0,
    timeout=120,
)

# Define the tools list
tools = [query_rul, lookup_manual, assess_health_status]

# Build the ReAct agent
agent = create_react_agent(llm, tools)

print(f"✅ Agent built successfully.")
print(f"  LLM:       {OLLAMA_MODEL}")
print(f"  Endpoint:  {OLLAMA_BASE_URL}")
print(f"  Tools:     {len(tools)} ({', '.join(t.name for t in tools)})")
print(f"\nThe agent is ready. It will use the ReAct loop to:")
print(f"  1. Reason about the maintenance question")
print(f"  2. Call tools (query RUL, lookup manual, assess health)")
print(f"  3. Observe tool results")
print(f"  4. Iterate until it can produce a recommendation")

✅ Agent built successfully.
  LLM:       qwen2.5:3b
  Endpoint:  http://localhost:11434/v1
  Tools:     3 (query_rul, lookup_manual, assess_health_status)

The agent is ready. It will use the ReAct loop to:
  1. Reason about the maintenance question
  2. Call tools (query RUL, lookup manual, assess health)
  3. Observe tool results
  4. Iterate until it can produce a recommendation


### Agent Architecture Diagram

```
┌─────────────────────────────────────────────────────────┐
│                    User Query                            │
    │              "Engine 1 shows degraded                   │
    │               performance. What should we do?"          │
    └──────────────────────┬────────────────────────────────┘
                         ▼
    ┌─────────────────────────────────────────────────────────┐
    │              LangGraph ReAct Agent                       │
    │                                                          │
    │  ┌──────────┐    ┌──────────────────────────┐           │
    │  │   LLM    │───▶│  Thought: What tools do   │           │
    │  │qwen2.5:3b    │ │  I need to call?         │           │
    │  └──────────┘    └──────────┬───────────────┘           │
    │                             ▼                           │
    │                    ┌──────────────────┐                 │
    │                    │  Action: Call    │                 │
    │                    │  query_rul(1)    │                 │
    │                    └────────┬─────────┘                 │
    │                             ▼                           │
    │              ┌──────────────────────────┐               │
    │              │  TOOL: query_rul         │               │
    │              │  → RUL: 35.2 cycles      │               │
    │              └────────┬─────────────────┘               │
    │                       ▼                                 │
    │              ┌──────────────────────────┐               │
    │              │  Observation: RUL=35.2   │               │
    │              │  → Need manual guidance  │               │
    │              └────────┬─────────────────┘               │
    │                       ▼                                 │
    │              ┌──────────────────────────┐               │
    │              │  Action: lookup_manual   │               │
    │              │  ("RUL decision")        │               │
    │              └────────┬─────────────────┘               │
    │                       ▼                                 │
    │              ┌──────────────────────────┐               │
    │              │  Observation: Manual     │               │
    │              │  says: Schedule B-Check  │               │
    │              └────────┬─────────────────┘               │
    │                       ▼                                 │
    │              ┌──────────────────────────┐               │
    │              │  Final Answer:            │               │
    │              │  Prescriptive             │               │
    │              │  Recommendation           │               │
    │              └──────────────────────────┘               │
    └─────────────────────────────────────────────────────────┘
    ```

## 7. Live Demo: Run the ReAct Loop

Now for the live demonstration. We will ask the agent a realistic maintenance question and watch it reason through the problem step by step.

**Scenario:** Engine 1 has been showing degraded performance in recent flights. The maintenance team wants to know what action to take.

The cell below will stream each step of the ReAct loop so you can see the agent's reasoning process in real time.

In [10]:
def run_agent_demo(user_query: str):
    """Run the agent with streaming output to show the ReAct loop step by step."""
    
    system_prompt = (
        "You assist a powerplant engineer for commercial turbofan engines. "
        "Your job is to help analyze engine health data and recommend maintenance actions. "
        "Use the available tools to query engine RUL predictions, look up maintenance "
        "procedures from the manual, and assess health status. "
        "Always base your recommendations on data from the tools, not on assumptions. "
        "Structure your final recommendation clearly with: (1) Current engine status, "
        "(2) Relevant maintenance procedures, (3) Specific recommended action, "
        "and (4) Timeframe for execution."
    )
    
    input_messages = {
        "messages": [
            ("system", system_prompt),
            ("user", user_query)
        ]
    }
    
    print(f"\n{'=' * 70}")
    print(f"USER QUERY: {user_query}")
    print(f"{'=' * 70}")
    print()
    
    step_count = 0
    for event in agent.stream(input_messages, stream_mode="values"):
        messages = event["messages"]
        last_msg = messages[-1]
        
        if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
            # Agent is calling a tool
            step_count += 1
            for tc in last_msg.tool_calls:
                print(f"━━━ STEP {step_count}: ACTION ━━━")
                print(f"  Tool:  {tc['name']}")
                print(f"  Args:  {tc['args']}")
                print()
        elif hasattr(last_msg, 'tool_call_id') and last_msg.tool_call_id:
            # Tool result (observation)
            step_count += 1
            print(f"━━━ STEP {step_count}: OBSERVATION ━━━")
            content = last_msg.content if isinstance(last_msg.content, str) else str(last_msg.content)[:500]
            # Truncate long observations for display
            if len(content) > 600:
                content = content[:600] + "..."
            print(f"  Result: {content}")
            print()
        elif last_msg.type == "ai" and (not hasattr(last_msg, 'tool_calls') or not last_msg.tool_calls):
            # Final answer (no tool calls = done)
            # But skip if we've already printed this message (first event)
            if step_count > 0 and last_msg.content and len(last_msg.content) > 50:
                print(f"{'=' * 70}")
                print("FINAL RECOMMENDATION")
                print(f"{'=' * 70}")
                print(last_msg.content)
                print()
                return last_msg.content
    
    return "Agent completed without a final answer."

In [11]:
# Run the live demo!
# Scenario: Engine 1 has been showing degraded performance.
demo_query = (
    "Engine 1 has been showing degraded performance in recent flights. "
    "What maintenance action do you recommend? "
    "Please check its RUL, look up the relevant maintenance procedures, "
    "and give me a clear recommendation."
)

final_recommendation = run_agent_demo(demo_query)


USER QUERY: Engine 1 has been showing degraded performance in recent flights. What maintenance action do you recommend? Please check its RUL, look up the relevant maintenance procedures, and give me a clear recommendation.

━━━ STEP 1: ACTION ━━━
  Tool:  query_rul
  Args:  {'engine_id': 1}

━━━ STEP 1: ACTION ━━━
  Tool:  assess_health_status
  Args:  {'rul_cycles': 0.5}

━━━ STEP 1: ACTION ━━━
  Tool:  lookup_manual
  Args:  {'query': 'inspection'}

━━━ STEP 2: OBSERVATION ━━━
  Result: Manual Section: 3. Inspection Intervals

Inspections are categorized by depth and scope. The interval is expressed in operating cycles.

| Inspection Level | Interval (cycles) | Scope | Downtime |
|---|---|---|---|
| **A-Check** | Every 150 cycles | Visual inspection, oil analysis, basic sensor calibration | 4–6 hours |
| **B-Check** | Every 600 cycles | Detailed sensor diagnostics, vibration analysis, borescope of fan and LPC | 1–2 days |
| **C-Check** | Every 3,600 cycles | Full engine teardown, HP

## 8. Second Demo: Comparing Multiple Engines

Let's run a second scenario to show the agent handling a different case. This time we'll ask about an engine that is in a more critical state.

In [12]:
# Second demo: A different engine with potentially different health status
demo_query_2 = (
    "I need a maintenance assessment for Engine 5. "
    "Check its RUL, look up the corrective actions for HPC degradation, "
    "and tell me what to do."
)

final_recommendation_2 = run_agent_demo(demo_query_2)


USER QUERY: I need a maintenance assessment for Engine 5. Check its RUL, look up the corrective actions for HPC degradation, and tell me what to do.

━━━ STEP 1: ACTION ━━━
  Tool:  query_rul
  Args:  {'engine_id': 5}

━━━ STEP 1: ACTION ━━━
  Tool:  lookup_manual
  Args:  {'query': 'HPC degradation'}

━━━ STEP 1: ACTION ━━━
  Tool:  assess_health_status
  Args:  {'rul_cycles': 100.5}

━━━ STEP 2: OBSERVATION ━━━
  Result: Health Assessment for RUL = 100.5 cycles:
  Status:  MONITOR
  Action:  Increase sensor monitoring frequency. Review trend data at next A-Check.

FINAL RECOMMENDATION
Based on the analysis, Engine 5 has a Remaining Useful Life (RUL) of approximately 100.3 operating cycles. The health assessment categorizes this as "MONITOR," indicating that it is currently in a condition where increased sensor monitoring and trend data review are recommended.

The relevant corrective actions for HPC degradation from the maintenance manual suggest performing a borescope inspection if

## 9. Troubleshooting

If you encounter issues with the agent:

**Ollama not running:**
- Check for the Ollama icon in your menu bar (macOS) or run `ollama serve` in a terminal.
- The pre-flight check (Section 2) will catch this and show a clear error.

**Model not found:**
- Run `ollama pull qwen2.5:3b` in a terminal.\n
- Verify with `ollama list | grep qwen2.5`.

**Agent takes too long:**

- The ReAct loop makes multiple LLM calls. Each call takes 10–30 seconds depending on your hardware.- `qwen2.5:3b` has been tested and works reliably for this demo.

- The `timeout=120` setting gives each call up to 2 minutes.- The LLM reads tool docstrings to understand when to call them. If the model is too small, it may skip tool calling.

- If you see timeout errors, try restarting Ollama or closing other apps.- Make sure the tool definitions (Section 5) were executed successfully.

**Agent doesn't call tools:**

In [13]:
# Troubleshooting: Rebuild the agent with a different model
# If you want to test with a different Ollama model, change the model name below.

# from langchain_openai import ChatOpenAI
# from langgraph.prebuilt import create_react_agent
#
# alt_llm = ChatOpenAI(
#     model="qwen2.5:3b",  # or any other model you have pulled
#     base_url=OLLAMA_BASE_URL,
#     api_key="not-needed",
#     temperature=0,
#     timeout=120,
# )
# alt_agent = create_react_agent(alt_llm, tools)

print("✅ All systems operational.")
print(f"   LLM:       {OLLAMA_MODEL}")
print(f"   Endpoint:  {OLLAMA_BASE_URL}")
print()
print("To switch to a different model:")
print(f"  1. Pull the model: ollama pull <model_name>")
print("  2. Uncomment the alt_llm block above")
print("  3. Replace 'agent' with 'alt_agent' in the demo cells")

✅ All systems operational.
   LLM:       qwen2.5:3b
   Endpoint:  http://localhost:11434/v1

To switch to a different model:
  1. Pull the model: ollama pull <model_name>
  2. Uncomment the alt_llm block above
  3. Replace 'agent' with 'alt_agent' in the demo cells


## 10. Summary and Key Takeaways

### What We Built

We built an **agentic AI system** for prescriptive maintenance that combines three elements:

| Component | Role | Technology |
|---|---|---|
| **Predictive Model** | Quantitative engine health assessment | Day 1 RUL model (sklearn) |
| **Domain Knowledge** | Maintenance procedures and thresholds | Engine Maintenance Program manual |
| **Agentic Orchestrator** | Reasoning, tool selection, recommendation | LLM + LangChain + LangGraph (ReAct loop) |

### The Agentic Pattern: Harness + LLM + Tools

- **The LLM alone** can only generate text — it cannot run computations or access data.
- **The tools alone** have no reasoning — they execute functions but don't know when to call them.
- **Together**, via the ReAct loop, they form an autonomous agent that can perceive, reason, act, and iterate.

### From Predictive to Prescriptive

| Analytics Level | Question | Our Demo |
|---|---|---|
| **Descriptive** | What happened? | Sensor trajectories (Day 1, Section 4) |
| **Diagnostic** | Why did it happen? | PCA health clusters (Day 1, Section 6) |
| **Predictive** | What will happen? | RUL prediction (Day 1, Section 8) |
| **Prescriptive** | What should we do? | **This demo** — agent recommendation |

### The Calculator Lesson — Revisited

Remember the calculator lesson from Day 1? The risk is not using the tool — it is losing the skill to check its work.

The same applies to agentic AI:
- The agent can query data, read manuals, and produce recommendations.
- **But the human maintenance engineer remains accountable** for the final decision.
- The agent is an assistant, not a replacement. It augments human judgment; it does not replace it.

### What's Next?

In production, this pattern scales:
- More tools (vibration analysis, oil debris monitoring, flight data recorder integration).
- More sophisticated RAG (retrieval-augmented generation) for larger manuals.
- Multi-agent systems (one agent for diagnostics, one for scheduling, one for parts logistics).
- Integration with maintenance planning systems (AMOS, TRAX, etc.).

But the core pattern remains the same: **Harness + LLM + Tools**.

And it all runs locally — no cloud dependency, no data leaves your machine.